In [ ]:
pip install spacy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
import spacy as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns
from sklearn.metrics import classification_report
import sklearn
from sklearn.datasets import fetch_20newsgroups
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

In [ ]:
df_train = fetch_20newsgroups(subset = 'train')

In [ ]:
print('TOPIC NAME:')
print(df_train.target_names[df_train.target[1]])

print('LABEL VALUE:')
print(df_train.target[1])

print('Data:')
print(df_train.data[1])

In [ ]:
plt.figure(figsize = (4, 4))
plt.hist(df_train)
plt.title('distribution of data')
plt.show()

In [ ]:
bins, counts = np.unique(df_train.target, return_counts = True)
freq_series = pd.Series(counts/len(df_train.data))
plt.figure(figsize = (8, 6))
ax = freq_series.plot(kind = 'bar')
ax.set_xticklabels(bins, rotation = 0)
plt.ylabel('Data present in each label')
plt.xlabel('Label Values')
plt.show()

In [ ]:
for i in range(len(df_train.target)):
    bins, counts = np.unique(df_train.target_names[df_train.target[i]], return_counts=True)
    freq_series = pd.Series(counts/len(df_train.data))
    plt.figure(figsize=(12, 8))
    ax = freq_series.plot(kind='bar')
    ax.set_xticklabels(bins, rotation=0)
    plt.ylabel('Data present in of each label')
    plt.xlabel('Label Values')
    plt.show()

In [ ]:
train_data, test_data, train_label, test_label = train_test_split(df_train.data, df_train.target, test_size = 0.2, random_state = 2)


In [ ]:
train_data[0]

In [ ]:
print(len(train_data))
print(len(test_data))

In [ ]:
nlp = sp.load('en_core_web_sm')

In [ ]:
nlp

In [ ]:
nlp.pipe_names

In [ ]:
nlp = sp.blank('en')

In [ ]:
nlp

In [ ]:
nlp.pipe_names

In [ ]:
def sp_tokenizer(doc):
    return [t.text for t in nlp(doc) if not t.is_punct and not t.is_space and t.is_alpha]

In [ ]:
vect = TfidfVectorizer(tokenizer = sp_tokenizer)
train_feature_vects = vect.fit_transform(train_data)

In [ ]:
vect = TfidfVectorizer(tokenizer = sp_tokenizer)
train_feature_vects = vect.fit_transform(train_data)

In [ ]:
NB = MultinomialNB()
NB.fit(train_feature_vects, train_label)

In [ ]:
NB.get_params()

In [ ]:
train_preds = NB.predict(train_feature_vects)

In [ ]:
metrics.f1_score(train_label, train_preds, average = 'macro')

In [ ]:
df_train_filtered = fetch_20newsgroups(subset = 'train', remove = ('headers', 'footers', 'quotes'))

In [ ]:
train_data_2, test_data_2, train_label_2, test_label_2 = train_test_split(df_train_filtered.data, df_train_filtered.target, test_size = 0.2, random_state = 1)

In [ ]:
vect = TfidfVectorizer(tokenizer = sp_tokenizer)
train_feature_vects = vect.fit_transform(train_data_2)

In [ ]:
NB = MultinomialNB()
NB.fit(train_feature_vects, train_label_2)

In [ ]:
NB.get_params()

In [ ]:
train_preds = NB.predict(train_feature_vects)

In [ ]:
metrics.f1_score(train_label_2, train_preds, average = 'macro')

In [ ]:
test_feature_vects = vect.transform(test_data_2)

In [ ]:
test_preds = NB.predict(test_feature_vects)

In [ ]:
metrics.f1_score(test_label_2, test_preds, average = 'macro')

In [ ]:
plt.figure(figsize = (8, 6))
cm = confusion_matrix(train_label, train_preds)
sns.heatmap(cm, annot = True, cmap = 'coolwarm')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6)) 
disp = ConfusionMatrixDisplay.from_estimator(NB, test_feature_vects, test_label, normalize='true', display_labels=df_train_filtered.target_names, xticks_rotation='vertical', ax=ax)

In [ ]:
cr = classification_report(test_label_2, test_preds, target_names = df_train_filtered.target_names)

In [ ]:
print(cr)

In [ ]:
len(train_data_2)

In [ ]:
len(train_feature_vects[0].toarray().flatten())

In [ ]:
nlp = sp.load('en_core_web_sm')

In [ ]:
unwanted_pipes = ['ner', 'parser']

# Further remove stop words and take the lemma instead of token text.
def sp_tokenizer(doc):
  with nlp.disable_pipes(*unwanted_pipes):
    return [t.lemma_ for t in nlp(doc) if not t.is_punct and not t.is_space and not t.is_stop and t.is_alpha]

In [ ]:
vect = TfidfVectorizer(tokenizer = sp_tokenizer)
train_feature_vects = vect.fit_transform(train_data_2)

In [ ]:
len(train_feature_vects[0].toarray().flatten())

In [ ]:
NB.fit(train_feature_vects, train_label_2)

In [ ]:
train_preds = NB.predict(train_feature_vects)

In [ ]:
metrics.f1_score(train_label_2, train_preds, average = 'macro')

In [ ]:
test_feature_vects = vect.transform(test_data_2)

In [ ]:
test_preds = NB.predict(test_feature_vects)

In [ ]:
metrics.f1_score(test_label_2, test_preds, average = 'macro')

In [ ]:
alphas = {'alpha' : [0.01, 0.1, 0.5, 1.0, 10.0], }

NB_grid = sklearn.model_selection.GridSearchCV(MultinomialNB(), param_grid = alphas, scoring = 'f1_macro', n_jobs = -1, cv = 5, verbose=5)
NB_grid.fit(train_feature_vects, train_label_2)

In [ ]:
NB_grid.best_params_

In [ ]:
NB_grid.best_estimator_

In [ ]:
Best_NB = NB_grid.best_estimator_

In [ ]:
pred_labels = Best_NB.predict(test_feature_vects)

In [ ]:
metrics.f1_score(test_label_2, pred_labels, average = 'macro')

In [ ]:
fig, ax = plt.subplots(figsize = (15, 15))
cm = ConfusionMatrixDisplay.from_estimator(Best_NB, test_feature_vects, test_label_2, normalize = 'true', display_labels = df_train_filtered.target_names, xticks_rotation = 'vertical', ax = ax)
plt.show()

In [ ]:
def show_top_words(classifier, vectorizer, categories, top_n):
    feature_names = np.asarray(vectorizer.get_feature_names_out())
    for i, category in enumerate(categories):
        prob_sorted = classifier.feature_log_prob_[i, :].argsort()[::-1]
        print("%s: %s" % (category, " ".join(feature_names[prob_sorted[:top_n]])))

In [ ]:
show_top_words(Best_NB, vect, df_train_filtered.target_names, 10)

In [ ]:
txt_clf = Pipeline([
    ('vect', TfidfVectorizer(tokenizer = sp_tokenizer)),
    ('Best_NB', MultinomialNB(alpha = 0.01))
])

In [ ]:
txt_clf.fit(df_train_filtered.data, df_train_filtered.target)

In [ ]:
df_test_filtered = fetch_20newsgroups(subset = 'test',  remove = ('headers', 'footers', 'quotes'))

In [ ]:
pred_vals = txt_clf.predict(df_test_filtered.data)

In [ ]:
print(metrics.classification_report(df_test_filtered.target, pred_vals, target_names = df_test_filtered.target_names))

In [ ]:
dir()

In [ ]:
new_text = """
NASA successfully launched a new satellite into space.
Scientists are studying black holes and galaxies.
"""

clean_text = preprocess(new_text)

X_new = vect.transform([clean_text])

prediction = Best_NB.predict(X_new)

print("Predicted Label:", prediction)

In [ ]:
news = fetch_20newsgroups(subset='train')

target_names = news.target_names

In [ ]:
print("Predicted Category:", target_names[prediction[0]])

In [ ]:
print(txt_clf)

In [ ]:
# Test the trained model

new_text = [
    "NASA successfully launched a new satellite into space. Scientists are studying galaxies and black holes."
]

prediction = txt_clf.predict(new_text)

print("Predicted Label:", prediction[0])

# Get category name
news = fetch_20newsgroups(subset='train')

print("Predicted Category:", news.target_names[prediction[0]])

In [ ]:
test_news = [
    "The government passed a new law in parliament.",
    "The hockey team won the championship after an exciting final.",
    "Microsoft released a new version of Windows.",
    "NASA discovered water on Mars.",
    "The baseball player scored a home run."
]

news = fetch_20newsgroups(subset='train')

for text in test_news:
    pred = txt_clf.predict([text])[0]

    print("="*60)
    print("Text:", text)
    print("Predicted Category:", news.target_names[pred])

In [ ]:
from sklearn.metrics import classification_report

predictions = txt_clf.predict(test_data)

news = fetch_20newsgroups(subset='test')

print(classification_report(
    test_label,
    predictions,
    target_names=news.target_names
))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

predictions = txt_clf.predict(test_data)

news = fetch_20newsgroups(subset='test')

ConfusionMatrixDisplay.from_predictions(
    test_label,
    predictions,
    display_labels=news.target_names,
    xticks_rotation=90,
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.show()